# InstanSeg: Custom Dataset Creation, Training, and Testing
This notebook outlines the end-to-end pipeline for utilizing InstanSeg for cell instance segmentation. It covers:
1. Environment Setup & Dependency Installation
2. Data Preprocessing & Custom `.pth` Dataset Creation
3. Model Fine-Tuning & Exporting
4. Testing

*Notebook last updated in May 2026. All new standards, versions, and requirements will need to be handled separately.*

In [ ]:
# Install the custom fork of InstanSeg
!git clone https://github.com/sonyalytv/instanseg_cryobiology.git

In [ ]:
!pip install -e /kaggle/working/instanseg_cryobiology

In [ ]:
# Install necessary external dependencies
!pip install monai
!pip install stardist
!pip install edt
!pip install fastremap


In [ ]:
!pip install --upgrade ipython ipykernel

**Restart the kernel**
* Choose `Run` in the upper menu
* Choose `Restart & clear cell outputs`

## 1. Dataset Creation

In [1]:
import torch
import torchvision
import instanseg

In [2]:
import os
import re
import torch
import numpy as np
import pandas as pd
import fastremap

from skimage.measure import label
from skimage.morphology import remove_small_objects

from pathlib import Path
from PIL import Image




# ============================================================
# CONFIG
# ============================================================

DATASET_SLUG = "datasets"  

IMG_DIR = Path("/kaggle/input/datasets/yehorlyndin/dataset-base/dataset_nuclei/images")
MASK_DIR = Path("/kaggle/input/datasets/yehorlyndin/dataset-base/dataset_nuclei/masks")

DATASET_NAME = "my_nuclei_dataset"

# First experiment: 512x512
RESIZE_TO = (512, 512)
SAVE_DATASET_NAME = "target_dataset_512.pth"
SPLIT_CSV_NAME = "splits_grouped_stratified_seed42.csv"

SEED = 42
GROUP_SIZE = 3

RANGES = {
    "range_0277_0321": (277, 321),
    "range_0560_0594": (560, 594),
    "range_0697_0788": (697, 788),
}


# ============================================================
# CHECK PATHS
# ============================================================

print("Images folder:", IMG_DIR)
print("Masks folder:", MASK_DIR)

assert IMG_DIR.exists(), f"Images folder not found: {IMG_DIR}"
assert MASK_DIR.exists(), f"Masks folder not found: {MASK_DIR}"


# ============================================================
# PIL RESAMPLING COMPATIBILITY
# ============================================================

try:
    RESAMPLE_IMAGE = Image.Resampling.LANCZOS
    RESAMPLE_MASK = Image.Resampling.NEAREST
except AttributeError:
    RESAMPLE_IMAGE = Image.LANCZOS
    RESAMPLE_MASK = Image.NEAREST


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def extract_index(path):
    """
    Extract numeric index from filename.
    Examples:
    0277.jpg -> 277
    nucleus_0277.jpg -> 277
    image_0697_field.jpg -> 697
    """
    numbers = re.findall(r"\d+", Path(path).stem)
    if not numbers:
        raise ValueError(f"No numeric index found in filename: {path.name}")
    return int(numbers[-1])


def get_range_name(index):
    """
    Assign each image to one of the known index ranges.
    """
    for range_name, (start, end) in RANGES.items():
        if start <= index <= end:
            return range_name
    return "outside_known_ranges"


def find_mask_for_image(img_path):
    """
    Finds corresponding PNG instance mask for a JPG image.

    Supports:
    0277.jpg -> 0277.png
    0277.jpg -> 0277_masks.png
    image_0277.jpg -> image_0277.png
    image_0277.jpg -> image_0277_masks.png

    Also falls back to matching by numeric index.
    """
    stem = img_path.stem

    direct_candidates = [
        MASK_DIR / f"{stem}.png",
        MASK_DIR / f"{stem}_masks.png",
        MASK_DIR / f"{stem.replace('_img', '')}.png",
        MASK_DIR / f"{stem.replace('_img', '')}_masks.png",
    ]

    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    # Fallback: match by numeric index
    img_index = extract_index(img_path)
    possible_masks = list(MASK_DIR.glob("*.png"))

    matching_masks = [
        mask_path for mask_path in possible_masks
        if extract_index(mask_path) == img_index
    ]

    if len(matching_masks) == 1:
        return matching_masks[0]

    if len(matching_masks) > 1:
        raise ValueError(
            f"More than one mask found for image {img_path.name}: "
            f"{[m.name for m in matching_masks]}"
        )

    raise FileNotFoundError(f"No PNG mask found for image: {img_path.name}")


def load_image_rgb(img_path, resize_to):
    """
    Loads JPG image, converts to RGB, resizes to fixed size.
    """
    img = Image.open(img_path).convert("RGB")
    img = img.resize(resize_to, RESAMPLE_IMAGE)
    return np.array(img).astype(np.uint8)


def load_instance_mask(mask_path, resize_to):
    """
    Loads binary PNG mask and converts it to approximate instance mask.

    Original mask:
    0 = background
    1 = nuclei foreground

    Converted mask:
    0 = background
    1 = nucleus instance 1
    2 = nucleus instance 2
    3 = nucleus instance 3
    ...
    """

    mask = np.array(Image.open(mask_path))

    if mask.ndim == 3:
        mask = mask[..., 0]

    # Convert to binary: everything above 0 is foreground
    binary_mask = mask > 0

    # Resize binary mask with NEAREST interpolation
    pil_mask = Image.fromarray(binary_mask.astype(np.uint8))
    pil_mask = pil_mask.resize(resize_to, RESAMPLE_MASK)
    binary_mask = np.array(pil_mask).astype(bool)

    # Optional: remove tiny noise objects
    binary_mask = remove_small_objects(binary_mask, min_size=10)

    # Connected components:
    # each separate white region becomes a different instance label
    instance_mask = label(binary_mask, connectivity=2)

    # Refit labels to 0, 1, 2, 3...
    instance_mask = fastremap.refit(instance_mask.astype(np.uint16))

    return instance_mask.astype(np.uint16)


def make_groups_for_range(files, group_size=3):
    """
    Groups neighbouring images by index.
    This reduces leakage between train/val/test.
    """
    files = sorted(files, key=extract_index)

    groups = []
    current_group = []
    previous_idx = None

    for img_path in files:
        current_idx = extract_index(img_path)

        should_start_new_group = False

        if len(current_group) >= group_size:
            should_start_new_group = True

        if previous_idx is not None and current_idx - previous_idx > group_size:
            should_start_new_group = True

        if should_start_new_group:
            groups.append(current_group)
            current_group = []

        current_group.append(img_path)
        previous_idx = current_idx

    if current_group:
        groups.append(current_group)

    return groups


def make_grouped_stratified_split(image_files, seed=42, group_size=3):
    """
    Creates grouped stratified split by known index ranges.

    Approximate target:
    Train      ~70%
    Validation ~10%
    Test       ~20%

    Similar neighbouring images stay in the same split.
    """
    rng = np.random.default_rng(seed)

    files_by_range = {}

    for img_path in image_files:
        idx = extract_index(img_path)
        range_name = get_range_name(idx)
        files_by_range.setdefault(range_name, []).append(img_path)

    final_splits = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    group_records = []

    for range_name, files in files_by_range.items():
        files = sorted(files, key=extract_index)
        groups = make_groups_for_range(files, group_size=group_size)

        # Shuffle groups, not individual images
        rng.shuffle(groups)

        n_groups = len(groups)

        n_train = int(round(n_groups * 0.70))
        n_val = int(round(n_groups * 0.10))

        # Make sure small ranges still get validation if possible
        if n_groups >= 5 and n_val == 0:
            n_val = 1

        # Keep at least one test group if possible
        if n_groups - n_train - n_val <= 0 and n_groups >= 3:
            n_train = max(1, n_train - 1)

        train_groups = groups[:n_train]
        val_groups = groups[n_train:n_train + n_val]
        test_groups = groups[n_train + n_val:]

        print(f"\n{range_name}")
        print("Images:", len(files))
        print("Groups:", len(groups))
        print("Train images:", sum(len(g) for g in train_groups))
        print("Validation images:", sum(len(g) for g in val_groups))
        print("Test images:", sum(len(g) for g in test_groups))

        for split_name, split_groups in [
            ("Train", train_groups),
            ("Validation", val_groups),
            ("Test", test_groups)
        ]:
            for group_id, group in enumerate(split_groups):
                for img_path in group:
                    final_splits[split_name].append(img_path)
                    group_records.append({
                        "split": split_name,
                        "range": range_name,
                        "group_id_inside_split": group_id,
                        "index": extract_index(img_path),
                        "image": img_path.name
                    })

    print("\nFINAL SPLIT SIZES")
    print("Train:", len(final_splits["Train"]))
    print("Validation:", len(final_splits["Validation"]))
    print("Test:", len(final_splits["Test"]))
    print("Total:", sum(len(v) for v in final_splits.values()))

    return final_splits, group_records


# ============================================================
# COLLECT JPG IMAGES
# ============================================================

image_files = sorted(list(IMG_DIR.glob("*.jpg")) + list(IMG_DIR.glob("*.JPG")), key=extract_index)
mask_files = sorted(list(MASK_DIR.glob("*.png")) + list(MASK_DIR.glob("*.PNG")), key=extract_index)

print("Number of JPG images found:", len(image_files))
print("Number of PNG masks found:", len(mask_files))

assert len(image_files) == 134, f"Expected 134 images, found {len(image_files)}"
assert len(mask_files) == 134, f"Expected 134 masks, found {len(mask_files)}"


# ============================================================
# MAKE SPLIT
# ============================================================

splits, group_records = make_grouped_stratified_split(
    image_files,
    seed=SEED,
    group_size=GROUP_SIZE
)


# ============================================================
# BUILD INSTANSEG DATASET
# ============================================================

Segmentation_Dataset = {
    "Train": [],
    "Validation": [],
    "Test": []
}

split_records = []

for split_name, files in splits.items():
    for img_path in files:
        mask_path = find_mask_for_image(img_path)

        image = load_image_rgb(img_path, RESIZE_TO)
        mask = load_instance_mask(mask_path, RESIZE_TO)

        if mask.max() == 0:
            print(f"WARNING: mask has no nuclei: {mask_path.name}")

        item = {
            "image": image,
            "nucleus_masks": mask,
            "parent_dataset": DATASET_NAME,
            "original_size": image.shape,
            "image_modality": "Brightfield",
            "file_name": str(img_path),
        }

        Segmentation_Dataset[split_name].append(item)

        split_records.append({
            "split": split_name,
            "range": get_range_name(extract_index(img_path)),
            "index": extract_index(img_path),
            "image": img_path.name,
            "mask": mask_path.name,
            "resize": f"{RESIZE_TO[0]}x{RESIZE_TO[1]}"
        })


# ============================================================
# SAVE DATASET AND SPLIT TABLE
# ============================================================

save_dataset_path = Path("/kaggle/working") / SAVE_DATASET_NAME
torch.save(Segmentation_Dataset, save_dataset_path)

split_csv_path = Path("/kaggle/working") / SPLIT_CSV_NAME
pd.DataFrame(split_records).sort_values(["split", "range", "index"]).to_csv(split_csv_path, index=False)

group_csv_path = Path("/kaggle/working") / "groups_grouped_stratified_seed42.csv"
pd.DataFrame(group_records).sort_values(["split", "range", "index"]).to_csv(group_csv_path, index=False)

print("\nSaved dataset to:", save_dataset_path)
print("Saved split CSV to:", split_csv_path)
print("Saved group CSV to:", group_csv_path)

print("\nDataset split check:")
for split_name in ["Train", "Validation", "Test"]:
    print(split_name, len(Segmentation_Dataset[split_name]))


# ============================================================
# QUICK SAMPLE CHECK
# ============================================================

sample = Segmentation_Dataset["Train"][0]

print("\nSample image shape:", sample["image"].shape)
print("Sample mask shape:", sample["nucleus_masks"].shape)
print("Sample mask dtype:", sample["nucleus_masks"].dtype)
print("Sample mask min:", sample["nucleus_masks"].min())
print("Sample mask max:", sample["nucleus_masks"].max())
print("First unique mask values:", np.unique(sample["nucleus_masks"])[:20])

Images folder: /kaggle/input/datasets/yehorlyndin/dataset-base/dataset_nuclei/images
Masks folder: /kaggle/input/datasets/yehorlyndin/dataset-base/dataset_nuclei/masks
Number of JPG images found: 134
Number of PNG masks found: 134

range_0277_0321
Images: 45
Groups: 15
Train images: 30
Validation images: 6
Test images: 9

range_0560_0594
Images: 32
Groups: 11
Train images: 24
Validation images: 3
Test images: 5

range_0697_0788
Images: 57
Groups: 20
Train images: 39
Validation images: 6
Test images: 12

FINAL SPLIT SIZES
Train: 93
Validation: 15
Test: 26
Total: 134

Saved dataset to: /kaggle/working/target_dataset_512.pth
Saved split CSV to: /kaggle/working/splits_grouped_stratified_seed42.csv
Saved group CSV to: /kaggle/working/groups_grouped_stratified_seed42.csv

Dataset split check:
Train 93
Validation 15
Test 26

Sample image shape: (512, 512, 3)
Sample mask shape: (512, 512)
Sample mask dtype: uint16
Sample mask min: 0
Sample mask max: 121
First unique mask values: [ 0  1  2  3  

In [3]:
import os
import torch
import torchvision
from pathlib import Path

# REQUIRED
%load_ext autoreload
%autoreload 2

# Configure InstanSeg environment paths
os.environ['INSTANSEG_RAW_DATASETS'] = os.path.abspath("/kaggle/working/instanseg_cryobiology/Raw_Datasets/")
os.environ['INSTANSEG_DATASET_PATH'] = os.path.abspath("/kaggle/working/instanseg_cryobiology/instanseg/datasets/")

# Create directories if they do not exist
if not os.path.exists(os.environ['INSTANSEG_RAW_DATASETS']):
    os.mkdir(os.environ['INSTANSEG_RAW_DATASETS'])

if not os.path.exists(os.environ['INSTANSEG_DATASET_PATH']):
    os.mkdir(os.environ['INSTANSEG_DATASET_PATH'])

print("Environment paths configured.")

Environment paths configured.



# Visikuls


In [13]:
import os
from pathlib import Path
import torch
import numpy as np
import fastremap
from skimage.measure import label
from skimage.morphology import remove_small_objects
from PIL import Image

# ============================================================
IMG_DIR = Path("/kaggle/input/datasets/yehorlyndin/visikuls-data/instanseg/good") 
MASK_DIR = Path("/kaggle/input/datasets/yehorlyndin/visikuls-data/instanseg/mask_groups")
OUTPUT_PTH = Path("/kaggle/working/visikuls_dataset.pth")

DATASET_NAME = "visikuls"
RESIZE_TO = (512, 512)


try:
    RESAMPLE_IMAGE = Image.Resampling.LANCZOS
    RESAMPLE_MASK = Image.Resampling.NEAREST
except AttributeError:
    RESAMPLE_IMAGE = Image.LANCZOS
    RESAMPLE_MASK = Image.NEAREST

# ============================================================
def load_image_rgb(img_path, resize_to):
    img = Image.open(img_path).convert("RGB")
    img = img.resize(resize_to, RESAMPLE_IMAGE)
    return np.array(img).astype(np.uint8) # Оставляем NumPy (H, W, C)

def load_instance_mask(mask_path, resize_to):
    mask = np.array(Image.open(mask_path))
    if mask.ndim == 3:
        mask = mask[..., 0]
    
    binary_mask = mask > 0
    pil_mask = Image.fromarray(binary_mask.astype(np.uint8)).resize(resize_to, RESAMPLE_MASK)
    binary_mask = np.array(pil_mask).astype(bool)

    binary_mask = remove_small_objects(binary_mask, min_size=10)
    instance_mask = label(binary_mask, connectivity=2)
    instance_mask = fastremap.refit(instance_mask.astype(np.uint16))

    return instance_mask.astype(np.uint16)

# ============================================================
print(f"📁 Ищем картинки в: {IMG_DIR}")
image_files = sorted([f for f in IMG_DIR.glob("*.*") if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.tif']])

test_data = []

for img_path in image_files:
    mask_name = img_path.stem + ".png"
    mask_path = MASK_DIR / mask_name
    
    if not mask_path.exists():
        print(f"⚠️ Пропуск: Маска не найдена для {img_path.name}")
        continue
        
  
    image_np = load_image_rgb(img_path, RESIZE_TO)
    mask_np = load_instance_mask(mask_path, RESIZE_TO)
    
    if mask_np.max() == 0:
        print(f"⚠️ Предупреждение: Маска пустая (нет клеток): {mask_path.name}")
        
    item = {
        "image": image_np,
        "nucleus_masks": mask_np,
        "parent_dataset": DATASET_NAME,
        "original_size": image_np.shape,
        "image_modality": "Brightfield",
        "file_name": str(img_path.name),
    }
    
    test_data.append(item)


final_dataset = {
    "Test": test_data
}

torch.save(final_dataset, OUTPUT_PTH)
print(f"\n✅ Успех! Датасет сохранен в {OUTPUT_PTH}")
print(f"В него вошло картинок: {len(test_data)}")

📁 Ищем картинки в: /kaggle/input/datasets/yehorlyndin/visikuls-data/instanseg/good

✅ Успех! Датасет сохранен в /kaggle/working/visikuls_dataset.pth
В него вошло картинок: 16


## 2. Model Training & Exporting

In this section, we will train our InstanSeg model on the dataset we just created. Training from scratch can take a long time, so it is common practice to start with pre-trained weights (Transfer Learning) or resume from a previous checkpoint. 

First, we will set up the directory for our model and move an existing checkpoint into it (simply skip this part if you want to train from scartch).

In [ ]:
from pathlib import Path
import shutil

model_folder_name = "augmented_dataset_512"
model_dir = Path(f"/kaggle/working/instanseg_cryobiology/instanseg/models/{model_folder_name}")
model_dir.mkdir(parents=True, exist_ok=True)

source_weights = "/kaggle/input/models/yehorlyndin/instanseg-aug/pytorch/default/1/model_weights_best.pth"
destination_weights = model_dir / "model_weights_best.pth"

# Copy the weights into our new model directory so training can resume from them
shutil.copy(source_weights, destination_weights)

### Initiating the Training Process

Now we call the `instanseg_training` function. This function handles the training loop behind the scenes. 
Here is a quick breakdown of the key parameters you might want to tweak:
* `target_segmentation="C"`: Tells the model we are segmenting Cells (use "N" for Nuclei).
* `num_epochs`: The maximum number of passes through the training data.
* `max_no_improvement`: Early stopping criterion. If the model doesn't improve for this many epochs, training stops to prevent overfitting.
* `hotstart_training`: The number of initial epochs to train before evaluating.

In [ ]:
import torch
from instanseg.scripts.train import instanseg_training

torch.cuda.empty_cache()

instanseg_training(
    data_path = "/kaggle/input/datasets/yehorlyndin/augmented-dataset-512/augmented_dataset_512.pth", 
    segmentation_dataset = Segmentation_Dataset, 
    source_dataset = "my_nuclei_dataset", 
    
    
    output_path = "/kaggle/working/instanseg_cryobiology/instanseg/models/",
    experiment_str = "augmented_dataset_512",
    
    target_segmentation = "N", 
    num_epochs = 35, 
    max_no_improvement = 5,
    batch_size = 4,
    lr = 0.0001,
   
    hotstart_training = 0
)

### Saving and Exporting the Model

Because Kaggle clears its `/kaggle/working/` directory when the session ends, we need to zip our newly trained model folder so it can be easily downloaded via the Kaggle interface.

We also use `export_to_torchscript`. TorchScript optimizes the PyTorch model, making it faster and allowing it to be deployed in production environments.

In [ ]:
!zip -r my_instanseg.zip "/kaggle/working/instanseg_cryobiology/instanseg/models/augmented_dataset_512"

In [ ]:
from instanseg.utils.utils import export_to_torchscript
import os

os.environ["INSTANSEG_MODEL_PATH"] = str(os.path.abspath("/kaggle/working/instanseg_cryobiology/instanseg/models"))

os.environ["INSTANSEG_TORCHSCRIPT_PATH"] = str(os.path.abspath("/kaggle/working"))

export_to_torchscript("augmented_dataset_512", show_example=False)

## 3. Testing

This part is independent from the previous ones. Load your trained model into kaggle: load `experiment_log.csv` and `model_weights_best.pth`. Also load the test dataset in `.pth` format, pay attention to the right splits.

To evaluate our model's performance, we use InstanSeg's built-in testing script. This script calculates metrics and can optionally save the visual overlays of the predictions.

**CLI Parameter Breakdown:**
* `-d_p`: Directory path containing the dataset to test.
* `-m_p`: Path to the trained model directory you want to evaluate.
* `-o_f`: Output folder for the test results.
* `-data`: Name of the specific `.pth` dataset file to evaluate against.
* `-save_ims True`: Saves visual images of the predicted masks overlaid on the input.
* `-target "C"`: Segmenting Cells.
* `-set "Test"`: Specifies which split of the `.pth` file to use (Train/Validation/Test).

In [14]:
%cd /kaggle/working/instanseg_cryobiology/instanseg/scripts

!python test.py \
-d_p "/kaggle/working" \
-m_p "/kaggle/input/models/yehorlyndin/instanseg-best-model/pytorch/default/1" \
-m_f "augmented_dataset_512" \
-o_f "/kaggle/working/test_results" \
-data "visikuls_dataset.pth" \
-save_ims True \
-target "N" \
-set "Test" \
-source "all"

/kaggle/working/instanseg_cryobiology/instanseg/scripts
<frozen importlib._bootstrap_external> (1301): The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-05-29 19:01:54.564953: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780081314.589819     475 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780081314.598043     475 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780081314.620016     475 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:178008

In [15]:
# Return to the root working directory
%cd /kaggle/working

# Zip the results folder so it can be downloaded and analyzed locally
!zip -r test_results.zip "/kaggle/working/test_results"

/kaggle/working
updating: kaggle/working/test_results/ (stored 0%)
updating: kaggle/working/test_results/images/ (stored 0%)
updating: kaggle/working/test_results/images/db_img_0169_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0605_overlay.png (deflated 2%)
updating: kaggle/working/test_results/images/db_img_0187_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0188_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0173_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0084_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0192_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0201_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/db_img_0601_overlay.png (deflated 2%)
updating: kaggle/working/test_results/images/db_img_0172_overlay.png (deflated 0%)
updating: kaggle/working/test_results/images/